In [ ]:
import logging
from importlib import reload
import os

import torch
import torch.nn as nn
import cv2
import numpy as np
from tqdm import trange

from pathlib import Path

from marmopose.version import __version__ as marmopose_version
from marmopose.config import Config
from marmopose.processing.prediction import Predictor
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import json


logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(name)s - %(message)s')
logger = logging.getLogger(__name__)

logger.info(f'MarmoPose version: {marmopose_version}')

# cage = 'etho'
cage = 'home'
CAP = cage[0].upper()
Cage = CAP + cage[1:]

n_cams = 4 if cage == 'etho' else 6

In [ ]:
config_path = '../configs/default.yaml'

# config = Config(
#     config_path=config_path,
    
#     n_tracks=1,
#     project='../demos/test',
#     det_model= '../data/detection_model_finetune_home_onlymarmoset',
#     pose_model= '../data/pose_model_finetune_home',

# )
# print(config.sub_directory)

# config_moreemptyframes = Config(
#     config_path=config_path,
    
#     n_tracks=1,
#     project='../demos/test',
#     det_model= '../data/detection_model_finetune_home_more_empty_frames',
#     pose_model= '../data/pose_model_finetune_home',

# )
# print(config.sub_directory)

config_finetune = Config(
    config_path=config_path,
    
    n_tracks=1,
    project='../demos/test',
    det_model= f'../data/detection_model_finetune_{cage}',
    pose_model= f'../data/pose_model_finetune_{cage}',

)
print(config_finetune.sub_directory)


config_base = Config(
    config_path=config_path,
    
    n_tracks=1,
    project='../demos/test',
)
print(config_base.sub_directory)

In [ ]:
predictor = Predictor(config_finetune, batch_size=4)
predictor_base = Predictor(config_base, batch_size=4)

In [ ]:
import json
dataset_dir = f'../../Sleap/TestData{Cage}With/marmoset_family/'
with open(os.path.join(dataset_dir,'annotations/all.json'), 'r') as f:
    test_json = json.load(f)
img_ids = [ann['image_id'] for ann in test_json['annotations']]
# img_ids_annotated = [ann['image_id'] for ann in test_json['annotations'] if 'keypoints' in ann.keys()]
# img_ids_unannotated = list(set(img_ids) - set(img_ids_annotated))
images = [cv2.imread(os.path.join(dataset_dir,'images',img['file_name'])) for img in test_json['images'] if img['id'] in img_ids]

gt_keypoints = []
gt_bboxes = []
for ann in test_json['annotations']:
    if 'keypoints' in ann.keys():
        gt_keypoints.append(ann['keypoints'])
    else:
        gt_keypoints.append([np.nan]*48)

    if 'bbox' in ann.keys():
        gt_bboxes.append(ann['bbox'])
    else:
        gt_bboxes.append([np.nan]*4)

gt_keypoints = np.array(gt_keypoints).reshape((-1,1,16,3))
gt_keypoints[gt_keypoints[:,:,:,2] == 0] = np.nan
gt_bboxes = np.array(gt_bboxes).reshape((-1,1,4))
gt_bboxes[:,:,2] = gt_bboxes[:,:,0] + gt_bboxes[:,:,2]
gt_bboxes[:,:,3] = gt_bboxes[:,:,1] + gt_bboxes[:,:,3]


In [ ]:
sorted_indices = np.load(f'../../Sleap/TestData{Cage}With/sorted_indices.npy')
missing_indices = np.load(f'../../Sleap/TestData{Cage}With/missing_indices.npy')
print(sorted_indices)
sorted_images = np.array(images)[sorted_indices]
sorted_gt_keypoints = gt_keypoints[sorted_indices]
sorted_gt_bboxes = gt_bboxes[sorted_indices]


In [ ]:
batch_size = 16
import gc
torch.cuda.empty_cache()
gc.collect()
points_with_score_2d_finetuned = np.empty((0,1,16,3))
bboxes_finetuned = np.empty((0,1,4))
points_with_score_2d_base = np.empty((0,1,16,3))
bboxes_base = np.empty((0,1,4))
for i in range(0,len(sorted_images),batch_size):
    points_with_score_2d_finetuned_part, bboxes_finetuned_part = predictor.predict_image_batch(sorted_images[i:i + batch_size])
    points_with_score_2d_base_part, bboxes_base_part = predictor_base.predict_image_batch(sorted_images[i:i + batch_size])
# torch.cuda.empty_cache()
# points_with_score_2d_finetuned_part2, bboxes_finetuned_part2 = predictor.predict_image_batch(sorted_images[100:])
# torch.cuda.empty_cache()
# points_with_score_2d_base_part1, bboxes_base_part1 = predictor_base.predict_image_batch(sorted_images[:100])
# torch.cuda.empty_cache()
# points_with_score_2d_base_part2, bboxes_base_part2 = predictor_base.predict_image_batch(sorted_images[100:])
# torch.cuda.empty_cache()
    points_with_score_2d_finetuned = np.concatenate((points_with_score_2d_finetuned,points_with_score_2d_finetuned_part), axis=0)
    bboxes_finetuned = np.concatenate((bboxes_finetuned,bboxes_finetuned_part), axis=0)
    points_with_score_2d_base = np.concatenate((points_with_score_2d_base,points_with_score_2d_base_part), axis=0)
    bboxes_base = np.concatenate((bboxes_base,bboxes_base_part), axis=0)
# points_with_score_2d_base = np.concatenate((points_with_score_2d_base_part1,points_with_score_2d_base_part2), axis=0)
# bboxes_base = np.concatenate((bboxes_base_part1,bboxes_base_part2), axis=0)
# points_with_score_2d_base[:,:,:,2] /= np.nanmax(points_with_score_2d_base[:,:,:,2])
# points_with_score_2d_finetuned[:,:,:,2] /= np.nanmax(points_with_score_2d_finetuned[:,:,:,2])


In [ ]:
def compute_true_false_positives(bboxes):
    frame_ids = np.arange(sorted_gt_bboxes.shape[0])
    frame_ids_absent = np.unique(np.nonzero(np.isnan(sorted_gt_bboxes))[0])
    frame_ids_present = np.setdiff1d(frame_ids, frame_ids_absent, assume_unique=True)
    truepositive = ~np.isnan(bboxes[frame_ids_present,0,0])
    falsepositive = ~np.isnan(bboxes[frame_ids_absent,0,0])
    print(f'False positives: {frame_ids_absent[np.nonzero(falsepositive)]}')
    print(f'False negatives: {frame_ids_present[np.nonzero(~truepositive)]}')
    perc_truepositive = 100 * (np.sum(truepositive)/frame_ids_present.size)
    perc_falsepositive = 100 * (np.sum(falsepositive)/frame_ids_absent.size)
    print(perc_truepositive, perc_falsepositive)
    return perc_truepositive, perc_falsepositive


perc_truepositives, perc_falsepositives  = list(zip(*[compute_true_false_positives(bboxes) for bboxes in (bboxes_base, bboxes_finetuned)]))

scores = ['True positives', 'False positives']
models = ['Base', 'Finetuned']
width = 1/(len(models) + 1)
offsets = np.arange(width + width/2, 1, width)
xs = np.arange(2)
for i, (tp, fp) in enumerate(zip(perc_truepositives, perc_falsepositives)):
    plt.bar(xs + offsets[i], [tp, fp], width, label = models[i],edgecolor = 'black')
plt.xticks(xs + 0.5 + width/2, labels=scores)
plt.yticks(np.linspace(0,100,5))
plt.ylabel('Percentage')
plt.legend()
plt.show()

In [ ]:
def compute_iou(bboxes):
    frame_ids_present_gt = np.unique(np.nonzero(~np.isnan(sorted_gt_bboxes))[0])
    frame_ids_present = np.unique(np.nonzero(~np.isnan(bboxes))[0])
    frame_ids_present_both = np.intersect1d(frame_ids_present_gt, frame_ids_present, assume_unique=True)
    bboxes_gt_present = sorted_gt_bboxes[frame_ids_present_both,0,:]
    bboxes_present = bboxes[frame_ids_present_both,0,:]
    assert np.sum(np.isnan(bboxes_gt_present)) == 0
    assert np.sum(np.isnan(bboxes_present)) == 0
    assert np.sum(bboxes_present[:,0] < bboxes_present[:,2]) == bboxes_present.shape[0]
    assert np.sum(bboxes_present[:,1] < bboxes_present[:,3]) == bboxes_present.shape[0]
    assert np.sum(bboxes_gt_present[:,0] < bboxes_gt_present[:,2]) == bboxes_gt_present.shape[0]
    assert np.sum(bboxes_gt_present[:,1] < bboxes_gt_present[:,3]) == bboxes_gt_present.shape[0]
    bboxes_inter = np.zeros_like(bboxes_present)
    bboxes_inter[:,:2] = np.max(np.concatenate((bboxes_present[:,:2,None],bboxes_gt_present[:,:2,None]),axis=2),axis=2)
    bboxes_inter[:,2:] = np.min(np.concatenate((bboxes_present[:,2:,None],bboxes_gt_present[:,2:,None]),axis=2),axis=2)
    area_inter = (bboxes_inter[:,2] - bboxes_inter[:,0]) * (bboxes_inter[:,3] - bboxes_inter[:,1])
    area_inter[bboxes_inter[:,0] > bboxes_inter[:,2]] = 0
    area_inter[bboxes_inter[:,1] > bboxes_inter[:,3]] = 0
    area1 = (bboxes_present[:,2] - bboxes_present[:,0]) * (bboxes_present[:,3] - bboxes_present[:,1])
    area2 = (bboxes_gt_present[:,2] - bboxes_gt_present[:,0]) * (bboxes_gt_present[:,3] - bboxes_gt_present[:,1])
    area_union = area1 + area2 - area_inter
    sidx = np.nonzero(area_union < 0)
    return area_inter/area_union

ious  = [compute_iou(bboxes) for bboxes in (bboxes_base, bboxes_finetuned)]

models = ['Base', 'Finetuned']
xs = np.arange(1, len(models) + 1)
for i in range(len(models)):
    plt.violinplot(ious[i:i+1], xs[i:i+1], showmeans=True)
plt.xticks(xs, labels=models)
plt.ylabel('IoU')
plt.show()
plt.boxplot(ious,showmeans=True,meanline=True)
plt.xticks(xs, labels=models)
plt.ylabel('IoU')
plt.show()

In [ ]:
example = 1
def compute_perc_correct_per_threshold_2D(bboxes, predicted, thresholds):
    frame_ids_present_gt = np.unique(np.nonzero(~np.isnan(sorted_gt_bboxes))[0])
    frame_ids_present = np.unique(np.nonzero(~np.isnan(bboxes))[0])
    frame_ids_present_both = np.intersect1d(frame_ids_present_gt, frame_ids_present, assume_unique=True)

    gt_present = sorted_gt_keypoints[frame_ids_present_both, ...]
    pred_present = predicted[frame_ids_present_both, ...]
    non_labelled_head_gt = np.sum(np.isnan(gt_present[:,:,:3,2]))
    labelled_head_gt = gt_present[:,:,:3,2].size - non_labelled_head_gt
    non_labelled_body_gt = np.sum(np.isnan(gt_present[:,:,[3,8],2]))
    labelled_body_gt = gt_present[:,:,[3,8],2].size - non_labelled_body_gt
    non_labelled_limbs_gt = np.sum(np.isnan(gt_present[:,:,np.r_[4:8,9:13],2]))
    labelled_limbs_gt = gt_present[:,:,np.r_[4:8,9:13],2].size - non_labelled_limbs_gt
    non_labelled_tail_gt = np.sum(np.isnan(gt_present[:,:,13:,2]))
    labelled_tail_gt = gt_present[:,:,13:,2].size - non_labelled_tail_gt
    error_head = (pred_present[:,:,:3,:2] - gt_present[:,:,:3,:2])
    error_head = np.sqrt(np.einsum('ijkl, ijkl -> ik',error_head,error_head))
    perc_head = [100 * np.sum(error_head < thresh)/labelled_head_gt for thresh in thresholds]

    error_body = (pred_present[:,:,[3,8],:2] - gt_present[:,:,[3,8],:2])
    error_body = np.sqrt(np.einsum('ijkl, ijkl -> ik',error_body,error_body))
    perc_body = [100 * np.sum(error_body < thresh)/labelled_body_gt for thresh in thresholds]

    error_limbs = (pred_present[:,:,np.r_[4:8,9:13],:2] - gt_present[:,:,np.r_[4:8,9:13],:2])
    error_limbs = np.sqrt(np.einsum('ijkl, ijkl -> ik',error_limbs,error_limbs))
    perc_limbs = [100 * np.sum(error_limbs < thresh)/labelled_limbs_gt for thresh in thresholds]

    error_tail = (pred_present[:,:,13:,:2] - gt_present[:,:,13:,:2])
    error_tail = np.sqrt(np.einsum('ijkl, ijkl -> ik',error_tail,error_tail))
    perc_tail = [100 * np.sum(error_tail < thresh)/labelled_tail_gt for thresh in thresholds]

    return perc_head, perc_body, perc_limbs, perc_tail

thresholds = np.arange(0,70,2)
allbboxes = [bboxes_base, bboxes_finetuned]
allpoints = [points_with_score_2d_base, points_with_score_2d_finetuned]

colors = ['r', 'b']
for i in range(len(models)):
    perc_head, perc_body, perc_limbs, perc_tail = compute_perc_correct_per_threshold_2D(allbboxes[i], allpoints[i], thresholds)
    plt.plot(thresholds,perc_head,c=colors[i],marker = 'o',ms=4, label = models[i] + ' head')
    plt.plot(thresholds,perc_body,c=colors[i],marker = 's',ms=4, label = models[i] + ' body')
    plt.plot(thresholds,perc_limbs,c=colors[i],marker = '^',ms=4, label = models[i] + ' limbs')
    plt.plot(thresholds,perc_tail,c=colors[i],marker = 'd',ms=4, label = models[i] + ' tail')
plt.legend()
plt.xlabel('Error threshold (pixels)')
plt.ylabel('Accuracy (%)')
plt.ylim((0,100))
plt.show()


# perc_head_finetuned, perc_body_finetuned, perc_limbs_finetuned, perc_tail_finetuned = compute_perc_correct_per_threshold_2D(sorted_gt_keypoints, points_with_score_2d_finetuned, thresholds)
# perc_head_base, perc_body_base, perc_limbs_base, perc_tail_base = compute_perc_correct_per_threshold_2D(sorted_gt_keypoints, points_with_score_2d_base, thresholds)
# plt.plot(thresholds,perc_head_finetuned,c='b',marker = 'o',ms=4)
# plt.plot(thresholds,perc_body_finetuned,c='b',marker = 's',ms=4)
# plt.plot(thresholds,perc_limbs_finetuned,c='b',marker = '^',ms=4)
# plt.plot(thresholds,perc_tail_finetuned,c='b',marker = 'd',ms=4)

# plt.plot(thresholds,perc_head_base,c='r',marker = 'o',ms=4)
# plt.plot(thresholds,perc_body_base,c='r',marker = 's',ms=4)
# plt.plot(thresholds,perc_limbs_base,c='r',marker = '^',ms=4)
# plt.plot(thresholds,perc_tail_base,c='r',marker = 'd',ms=4)

# n_ns = len(points_with_score_2d_dict.keys())
# for i, n in enumerate(sorted(points_with_score_2d_dict.keys())):
#     perc_head, perc_body, perc_limbs, perc_tail = compute_perc_correct_per_threshold_2D(sorted_gt_keypoints, points_with_score_2d_dict[n], thresholds)
#     plt.plot(thresholds,perc_head,c=((n_ns - i)/(n_ns + 1),(n_ns - i)/(n_ns + 1),1),marker = 'o',ms=4)
#     plt.plot(thresholds,perc_body,c=((n_ns - i)/(n_ns + 1),(n_ns - i)/(n_ns + 1),1),marker = 's',ms=4)
#     plt.plot(thresholds,perc_limbs,c=((n_ns - i)/(n_ns + 1),(n_ns - i)/(n_ns + 1),1),marker = '^',ms=4)
#     plt.plot(thresholds,perc_tail,c=((n_ns - i)/(n_ns + 1),(n_ns - i)/(n_ns + 1),1),marker = 'd',ms=4)



# plt.xlabel('Error threshold (pixels)')
# plt.ylabel('Accuracy (%)')
# plt.ylim((0,100))
# plt.show()

In [ ]:
print(points_with_score_2d_finetuned.shape)
print(bboxes_finetuned.shape)
print(points_with_score_2d_base.shape)
print(bboxes_base.shape)
print(gt_keypoints.shape)
print(gt_bboxes.shape)
print(np.nansum((sorted_gt_bboxes - bboxes_base)**2)/(sorted_gt_keypoints.shape[0] * sorted_gt_keypoints.shape[2]))
print(np.nansum((sorted_gt_bboxes - bboxes_finetuned)**2)/(sorted_gt_keypoints.shape[0] * sorted_gt_keypoints.shape[2]))
jitter = 0
for i, image in enumerate(sorted_images[jitter:jitter+20]):
    i = i + jitter
    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(20, 10))
    axes[0].set_title(img_ids[i])
    axes[0].imshow(image[:,:,[2,1,0]])
    axes[1].imshow(image[:,:,[2,1,0]])
    rect = patches.Rectangle(
        (sorted_gt_bboxes[i,0,0], sorted_gt_bboxes[i,0,1]), sorted_gt_bboxes[i,0,2] - sorted_gt_bboxes[i,0,0], sorted_gt_bboxes[i,0,3] - sorted_gt_bboxes[i,0,1],
        linewidth=1, edgecolor='g', facecolor='none'
    )
    axes[0].add_patch(rect)
    axes[0].scatter(sorted_gt_keypoints[i,0,:,0],sorted_gt_keypoints[i,0,:,1],c='g',s = 1)
    base_nans = np.isnan(points_with_score_2d_base[i,0,:,2])
    if not np.sum(base_nans) == 16:
        axes[0].scatter(points_with_score_2d_base[i,0,~base_nans,0],points_with_score_2d_base[i,0,~base_nans,1],c='r',s = points_with_score_2d_base[i,0,~base_nans,2])
    rect = patches.Rectangle(
        (sorted_gt_bboxes[i,0,0], sorted_gt_bboxes[i,0,1]), sorted_gt_bboxes[i,0,2] - sorted_gt_bboxes[i,0,0], sorted_gt_bboxes[i,0,3] - sorted_gt_bboxes[i,0,1],
        linewidth=1, edgecolor='g', facecolor='none'
    )
    
    axes[1].add_patch(rect)
    axes[1].scatter(sorted_gt_keypoints[i,0,:,0],sorted_gt_keypoints[i,0,:,1],c='g',s = 1)
    finetuned_nans = np.isnan(points_with_score_2d_finetuned[i,0,:,2])
    axes[1].scatter(points_with_score_2d_finetuned[i,0,~finetuned_nans,0],points_with_score_2d_finetuned[i,0,~finetuned_nans,1],c='b',s = points_with_score_2d_finetuned[i,0,~finetuned_nans,2])
    rect = patches.Rectangle(
        (bboxes_base[i,0,0], bboxes_base[i,0,1]), bboxes_base[i,0,2]-bboxes_base[i,0,0], bboxes_base[i,0,3]-bboxes_base[i,0,1],
        linewidth=1, edgecolor='r', facecolor='none'
    )
    axes[0].add_patch(rect)
    rect = patches.Rectangle(
        (bboxes_finetuned[i,0,0], bboxes_finetuned[i,0,1]), bboxes_finetuned[i,0,2]-bboxes_finetuned[i,0,0], bboxes_finetuned[i,0,3]-bboxes_finetuned[i,0,1],
        linewidth=1, edgecolor='b', facecolor='none'
    )
    axes[1].add_patch(rect)

    for bodyparts in config_base.visualization['skeleton'][::-1]:
        idx_bodyparts = []
        for bodypart in bodyparts:
            idx_bodyparts.append(config_base.animal['bodyparts'].index(bodypart))
        axes[0].plot(sorted_gt_keypoints[i,0,idx_bodyparts,0],sorted_gt_keypoints[i,0,idx_bodyparts,1], lw=1,color='g')
        axes[1].plot(sorted_gt_keypoints[i,0,idx_bodyparts,0],sorted_gt_keypoints[i,0,idx_bodyparts,1], lw=1,color='g')
        axes[0].plot(points_with_score_2d_base[i,0,idx_bodyparts,0],points_with_score_2d_base[i,0,idx_bodyparts,1], lw=1,color='r')
        axes[1].plot(points_with_score_2d_finetuned[i,0,idx_bodyparts,0],points_with_score_2d_finetuned[i,0,idx_bodyparts,1], lw=1,color='b')

    axes[0].axis('off')
    axes[1].axis('off')
    fig.show()




In [ ]:
# start_indices = np.concatenate((np.array([0]), 1 + missing_indices))
# end_indices = np.concatenate((missing_indices,np.array([sorted_indices.size + missing_indices.size])))

# sorted_images_with_missing = np.full((sorted_indices.size + missing_indices.size, *sorted_images.shape[1:]), 255)
# sorted_gt_keypoints_with_missing = np.full((sorted_indices.size + missing_indices.size, *gt_keypoints.shape[1:]), np.nan)
# sorted_base_keypoints_with_missing = np.full((sorted_indices.size + missing_indices.size, *points_with_score_2d_base.shape[1:]), np.nan)
# sorted_finetuned_keypoints_with_missing = np.full((sorted_indices.size + missing_indices.size, *points_with_score_2d_finetuned.shape[1:]), np.nan)
# for i, (idx0, idx1) in enumerate(zip(start_indices,end_indices)):
#     print(i,idx0,idx1)
#     sorted_images_with_missing[idx0:idx1, ...] = sorted_images[idx0-i:idx1-i, ...]
#     sorted_gt_keypoints_with_missing[idx0:idx1, ...] = sorted_gt_keypoints[idx0 -i:idx1 - i, ...]
#     sorted_base_keypoints_with_missing[idx0:idx1, ...] = points_with_score_2d_base[idx0 -i:idx1 - i, ...]
#     sorted_finetuned_keypoints_with_missing[idx0:idx1, ...] = points_with_score_2d_finetuned[idx0 -i:idx1 - i, ...]

# sorted_images_with_missing = sorted_images_with_missing.reshape(4,-1,*sorted_images_with_missing.shape[1:])
# sorted_gt_keypoints_with_missing = sorted_gt_keypoints_with_missing.reshape(4,-1,*sorted_gt_keypoints_with_missing.shape[2:])
# sorted_gt_keypoints_with_missing[sorted_gt_keypoints_with_missing[:,:,:,2] == 0,:] = np.nan
# print(sorted_gt_keypoints_with_missing.shape)
# sorted_base_keypoints_with_missing = sorted_base_keypoints_with_missing.reshape(4,-1,*sorted_base_keypoints_with_missing.shape[2:])
# sorted_finetuned_keypoints_with_missing = sorted_finetuned_keypoints_with_missing.reshape(4,-1,*sorted_finetuned_keypoints_with_missing.shape[2:])

In [ ]:
from tqdm import trange
from marmopose.calibration.cameras import CameraGroup

if cage == 'etho':
    camera_groups = CameraGroup.load_from_json(os.path.join('/srv','MarmOT','VideoTracking','Videos','CalibEtho','camera_params.json'))

elif cage == 'home':
    camera_groups = [CameraGroup.load_from_json(os.path.join('/srv','MarmOT','VideoTracking','Videos',f'TestHomeWithEtho{i}.1','Calib_preprocessed','camera_params.json')) for i in range(1,3)]
    
def triangulate_frame(camera_group, points_with_score_2d: np.ndarray, ransac=True):
    """
    Args:
        camera_group: CameraGroup
        points_with_score_2d: (n_cams, n_tracks, n_bodyparts, (x, y, score))
    
    Returns:
        points_3d: (n_bodyparts, (x, y, z))
    """


    if ransac:
        points_3d = camera_group.triangulate_ransac(points_with_score_2d, undistort=True)
    else:
        points_3d = camera_group.triangulate(points_with_score_2d, undistort=True)
        
    return points_3d

n_frames_per_cam = int(sorted_indices.size/n_cams)
diff_indices_cam1 = sorted_indices[1:n_frames_per_cam] -  sorted_indices[:n_frames_per_cam - 1]
sessions_for_frames = np.zeros((n_frames_per_cam), dtype = np.uint8)
for i, idx in enumerate(np.nonzero(diff_indices_cam1 != 1)[0] + 1):
    sessions_for_frames[idx:] = i + 1

points_with_score_2d_gt_reshaped = sorted_gt_keypoints.reshape(n_cams, -1, *sorted_gt_keypoints.shape[2:])
points_with_score_2d_base_reshaped = points_with_score_2d_base.reshape(n_cams, -1, *points_with_score_2d_base.shape[2:])
points_with_score_2d_finetuned_reshaped = points_with_score_2d_finetuned.reshape(n_cams, -1, *points_with_score_2d_finetuned.shape[2:])

n_cams, n_frames, n_bodyparts, n_dim = points_with_score_2d_gt_reshaped.shape
gt_points_3d = np.full((n_frames, n_bodyparts, 3), np.nan)
base_points_3d = np.full((n_frames, n_bodyparts, 3), np.nan)
finetuned_points_3d = np.full((n_frames, n_bodyparts, 3), np.nan)

for frame_idx in trange(n_frames, ncols=100, desc='Triangulating... ', unit='frames'):
    gt_all_points_with_score_2d_frame = points_with_score_2d_gt_reshaped[:, frame_idx]
    base_all_points_with_score_2d_frame = points_with_score_2d_base_reshaped[:, frame_idx]
    finetuned_all_points_with_score_2d_frame = points_with_score_2d_finetuned_reshaped[:, frame_idx]
    
    if isinstance(camera_groups, list):
        camera_group = camera_groups[sessions_for_frames[frame_idx]]
    else:
        camera_group = camera_groups
    
    gt_point_3d = triangulate_frame(camera_group, gt_all_points_with_score_2d_frame, ransac=True) 
    base_point_3d = triangulate_frame(camera_group, base_all_points_with_score_2d_frame, ransac=True) 
    finetuned_point_3d = triangulate_frame(camera_group, finetuned_all_points_with_score_2d_frame, ransac=True) 
        
    gt_points_3d[frame_idx] = gt_point_3d
    base_points_3d[frame_idx] = base_point_3d
    finetuned_points_3d[frame_idx] = finetuned_point_3d



In [ ]:
def compute_perc_correct_per_threshold_3D(groundtruth, predicted, thresholds):
    gt_head = groundtruth[:,:3,:]
    non_labelled_head_gt = np.sum(np.isnan(gt_head[:,:,2]))
    labelled_head_gt = gt_head[:,:,2].size - non_labelled_head_gt
    gt_body = groundtruth[:,[3,8],:]
    non_labelled_body_gt = np.sum(np.isnan(gt_body[:,:,2]))
    labelled_body_gt = gt_body[:,:,2].size - non_labelled_body_gt
    gt_limbs = groundtruth[:,np.r_[4:8,9:13],:]
    non_labelled_limbs_gt = np.sum(np.isnan(gt_limbs[:,:,2]))
    labelled_limbs_gt = gt_limbs[:,:,2].size - non_labelled_limbs_gt
    gt_tail = groundtruth[:,13:,:]
    non_labelled_tail_gt = np.sum(np.isnan(gt_tail[:,:,2]))
    labelled_tail_gt = gt_tail[:,:,2].size - non_labelled_tail_gt

    error_head_finetuned = (predicted[:,:3,:] - gt_head)
    error_head_finetuned = np.sqrt(np.einsum('ijk, ijk -> ij',error_head_finetuned,error_head_finetuned))
    perc_head_finetuned = [100 * np.sum(error_head_finetuned < thresh)/labelled_head_gt for thresh in thresholds]

    error_body_finetuned = (predicted[:,[3,8],:] - gt_body)
    error_body_finetuned = np.sqrt(np.einsum('ijk, ijk -> ij',error_body_finetuned,error_body_finetuned))
    perc_body_finetuned = [100 * np.sum(error_body_finetuned < thresh)/labelled_body_gt for thresh in thresholds]

    error_limbs_finetuned = (predicted[:,np.r_[4:8,9:13],:] - gt_limbs)
    error_limbs_finetuned = np.sqrt(np.einsum('ijk, ijk -> ij',error_limbs_finetuned,error_limbs_finetuned))
    perc_limbs_finetuned = [100 * np.sum(error_limbs_finetuned < thresh)/labelled_limbs_gt for thresh in thresholds]

    error_tail_finetuned = (predicted[:,13:,:] - gt_tail)
    error_tail_finetuned = np.sqrt(np.einsum('ijk, ijk -> ij',error_tail_finetuned,error_tail_finetuned))
    perc_tail_finetuned = [100 * np.sum(error_tail_finetuned < thresh)/labelled_tail_gt for thresh in thresholds]

    return perc_head_finetuned, perc_body_finetuned, perc_limbs_finetuned, perc_tail_finetuned


In [ ]:
thresholds = np.arange(0,101,4)

perc_head_finetuned, perc_body_finetuned, perc_limbs_finetuned, perc_tail_finetuned = compute_perc_correct_per_threshold_3D(gt_points_3d, finetuned_points_3d, thresholds)
perc_head_base, perc_body_base, perc_limbs_base, perc_tail_base = compute_perc_correct_per_threshold_3D(gt_points_3d, base_points_3d, thresholds)
plt.plot(thresholds,perc_head_finetuned,c='b',marker = 'o',ms=4)
plt.plot(thresholds,perc_body_finetuned,c='b',marker = 's',ms=4)
plt.plot(thresholds,perc_limbs_finetuned,c='b',marker = '^',ms=4)
plt.plot(thresholds,perc_tail_finetuned,c='b',marker = 'd',ms=4)

plt.plot(thresholds,perc_head_base,c='r',marker = 'o',ms=4)
plt.plot(thresholds,perc_body_base,c='r',marker = 's',ms=4)
plt.plot(thresholds,perc_limbs_base,c='r',marker = '^',ms=4)
plt.plot(thresholds,perc_tail_base,c='r',marker = 'd',ms=4)
plt.xlabel('Error threshold (mm)')
plt.ylabel('Accuracy (%)')
plt.ylim((0,100))
plt.show()

In [ ]:
xlim, ylim, zlim, _ = (1200, 730, 900, 30) if cage == 'home' else (660, 560, 800, 30)

for i in range(20,30):
    # Create a 3D scatter plot
    fig = plt.figure(figsize=(10,5))
    ax1 = fig.add_subplot(131, projection='3d')
    ax2 = fig.add_subplot(132, projection='3d')
    ax3 = fig.add_subplot(133, projection='3d')
    for bodyparts in config_base.visualization['skeleton'][::-1]:
        idx_bodyparts = []
        for bodypart in bodyparts:
            idx_bodyparts.append(config_base.animal['bodyparts'].index(bodypart))
        ax1.plot(gt_points_3d[i,idx_bodyparts,0],gt_points_3d[i,idx_bodyparts,1],gt_points_3d[i,idx_bodyparts,2], marker = 'o', ms=3,color='g')
        ax2.plot(base_points_3d[i,idx_bodyparts,0],base_points_3d[i,idx_bodyparts,1],base_points_3d[i,idx_bodyparts,2], marker = 'o', ms=3,color='r')
        ax3.plot(finetuned_points_3d[i,idx_bodyparts,0],finetuned_points_3d[i,idx_bodyparts,1],finetuned_points_3d[i,idx_bodyparts,2], marker = 'o', ms=3,color='b')
    ax1.set_title(f'frame {i} ground truth')
    ax1.set_xlim((0,xlim))
    ax1.set_ylim((0,ylim))
    ax1.set_zlim((0,zlim))
    ax2.set_title(f'frame {i} base model')
    ax2.set_xlim((0,xlim))
    ax2.set_ylim((0,ylim))
    ax2.set_zlim((0,zlim))
    ax3.set_title(f'frame {i} finetuned model')
    ax3.set_xlim((0,xlim))
    ax3.set_ylim((0,ylim))
    ax3.set_zlim((0,zlim))

In [ ]:
from marmopose.utils.data_io import load_points_3d_h5
with open('../../Sleap/TestDataEthoWith/frames_dict.json') as f:
    frames_in_videos = json.load(f)
print(frames_in_videos)
fullworkflow_finetuned_points3d = np.full((finetuned_points_3d.shape), np.nan)
fullworkflow_base_points3d = np.full((finetuned_points_3d.shape), np.nan)
i = 0
for video in sorted(frames_in_videos.keys()):
    points_3d = load_points_3d_h5(f"../../Videos/TestEthoWithHome{int(video)}.1/Output/points_3d/optimized.h5")
    fullworkflow_finetuned_points3d[i:i+len(frames_in_videos[video]),:,:] = points_3d[0,frames_in_videos[video],:,:]
    # points_3d = load_points_3d_h5(f"../../Videos/TestEthoWithHome{int(video)}.1/Output_basemodel/points_3d/optimized.h5")
    # fullworkflow_base_points3d[i:i+len(frames_in_videos[video]),:,:] = points_3d[0,frames_in_videos[video],:,:]
    i += len(frames_in_videos[video])


In [ ]:
thresholds = np.arange(0,70,2)
perc_head_fullworkflow_finetuned, perc_body_fullworkflow_finetuned, perc_limbs_fullworkflow_finetuned, perc_tail_fullworkflow_finetuned = compute_perc_correct_per_threshold_3D(gt_points_3d, fullworkflow_finetuned_points3d, thresholds)
perc_head_fullworkflow_base, perc_body_fullworkflow_base, perc_limbs_fullworkflow_base, perc_tail_fullworkflow_base = compute_perc_correct_per_threshold_3D(gt_points_3d, fullworkflow_base_points3d, thresholds)


plt.plot(thresholds,perc_head_fullworkflow_finetuned,c='b',marker = 'o',ms=4)
plt.plot(thresholds,perc_body_fullworkflow_finetuned,c='b',marker = 's',ms=4,)
plt.plot(thresholds,perc_limbs_fullworkflow_finetuned,c='b',marker = '^',ms=4)
plt.plot(thresholds,perc_tail_fullworkflow_finetuned,c='b',marker = 'd',ms=4)

# plt.plot(thresholds,perc_head_fullworkflow_base,c='r',marker = 'o',ms=4)
# plt.plot(thresholds,perc_body_fullworkflow_base,c='r',marker = 's',ms=4,)
# plt.plot(thresholds,perc_limbs_fullworkflow_base,c='r',marker = '^',ms=4)
# plt.plot(thresholds,perc_tail_fullworkflow_base,c='r',marker = 'd',ms=4)

plt.xlabel('Error threshold (mm)')
plt.ylabel('Accuracy (%)')
plt.ylim((0,100))
plt.show()

In [ ]:
xlim, ylim, zlim, _ = config.visualization['room_dimensions']

for i in range(30):
    # Create a 3D scatter plot
    fig = plt.figure(figsize=(10,5))
    ax1 = fig.add_subplot(131, projection='3d')
    ax2 = fig.add_subplot(132, projection='3d')
    ax3 = fig.add_subplot(133, projection='3d')
    for bodyparts in config.visualization['skeleton'][::-1]:
        idx_bodyparts = []
        for bodypart in bodyparts:
            idx_bodyparts.append(config.animal['bodyparts'].index(bodypart))
        ax1.plot(gt_points_3d[i,idx_bodyparts,0],gt_points_3d[i,idx_bodyparts,1],gt_points_3d[i,idx_bodyparts,2], marker = 'o', ms=3,color='g')
        ax2.plot(finetuned_points_3d[i,idx_bodyparts,0],finetuned_points_3d[i,idx_bodyparts,1],finetuned_points_3d[i,idx_bodyparts,2], marker = 'o', ms=3,color='b')
        ax3.plot(fullworkflow_finetuned_points3d[i,idx_bodyparts,0],fullworkflow_finetuned_points3d[i,idx_bodyparts,1],fullworkflow_finetuned_points3d[i,idx_bodyparts,2], marker = 'o', ms=3,color='indigo')
    ax1.set_title(f'frame {i} ground truth')
    ax1.set_xlim((0,xlim))
    ax1.set_ylim((0,ylim))
    ax1.set_zlim((0,zlim))
    ax2.set_title(f'frame {i} finetuned model')
    ax2.set_xlim((0,xlim))
    ax2.set_ylim((0,ylim))
    ax2.set_zlim((0,zlim))
    ax3.set_title(f'frame {i} fullworkflow finetuned model')
    ax3.set_xlim((0,xlim))
    ax3.set_ylim((0,ylim))
    ax3.set_zlim((0,zlim))